# 08 — PyTorch and MPS: the failures that never raise an exception

## The question, in one sentence

**Every real defect this project has hit in eighteen months of combined work produced wrong
numbers with no error message at all** — not a crash, not a warning, a plausible number sitting
quietly in the wrong place. This notebook is a craft notebook, not a motion-generation notebook:
it exists to teach the specific PyTorch and Apple-Silicon-MPS habits that catch this class of bug
before it costs a week, using this project's own real specimens plus a few more worth knowing.

**Every claim here is demonstrated by a runnable cell, wrong way next to right way, real numbers
printed.** Nothing is described without being shown. Anything that did not reproduce under direct
testing was dropped — the notebook says so, rather than keeping a plausible claim untested.

This notebook is deliberately larger than the others in this project: craft fluency needs
breadth, not a single finding.

## The intuition

A silent failure is not a bug that hides. It is a bug that produces a *completely reasonable*
number. The loss goes down. The shapes match (just not the shapes you meant). The code runs to
completion. Nothing in the output looks wrong, because the computer did exactly what you told it
to do — the gap is between what you told it and what you meant, and nothing in Python or PyTorch
is obligated to notice that gap for you.

This is why every trap below is followed immediately by a **habit** (Part 3) that would have
caught it. The traps are the disease; the habits are the immune system. A programmer who knows
one trap has learned one fact. A programmer who has internalized the habits catches traps they
have never personally seen before.

## Setup

In [1]:
import time
import warnings

import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
MPS_AVAILABLE = torch.backends.mps.is_available()
print(f"torch: {torch.__version__}, MPS available: {MPS_AVAILABLE}")

torch: 2.13.0, MPS available: True


# Part 1 — Foundations that make the later bugs possible

Not filler. Each one below is the specific mechanism behind at least one real bug in Part 2 —
read these first and Part 2 stops looking like a list of unrelated accidents.

## 1.1 Broadcasting — the single most valuable cell in this notebook

PyTorch (like NumPy) will "helpfully" expand mismatched shapes to make an operation work, rather
than raising an error. A `(N,)` tensor and a `(N,1)` tensor do not refuse to subtract — they
broadcast to `(N,N)`, comparing every prediction against every target instead of prediction *i*
against target *i*. The result trains, decreases, and looks fine.

In [2]:
N = 8
pred = torch.randn(N)          # shape (N,)  -- e.g. a model's N predictions
target = torch.randn(N, 1)     # shape (N,1) -- a common shape from a dataloader that
                                # forgot to .squeeze(-1), or a label column from pandas

wrong = pred - target           # broadcasts (N,) and (N,1) up to (N,N) -- no error, no warning
right = pred - target.squeeze(-1)

print(f"pred.shape={tuple(pred.shape)}  target.shape={tuple(target.shape)}")
print(f"(pred - target).shape = {tuple(wrong.shape)}   <-- {wrong.numel()} pairs, not {N}")
print(f"(pred - target.squeeze(-1)).shape = {tuple(right.shape)}   <-- {right.numel()} pairs, correct")
print()
print(f"'loss' from the WRONG shape (averages over all {wrong.numel()} cross-pairs): "
      f"{(wrong**2).mean().item():.4f}")
print(f"loss from the RIGHT shape (averages over the intended {N} pairs):        "
      f"{(right**2).mean().item():.4f}")

# the wrong version isn't just numerically different -- it trains, silently, on the wrong thing
p = pred.clone().requires_grad_(True)
((p - target) ** 2).mean().backward()
print(f"\nbackward() on the WRONG shape succeeds with no error, no warning: "
      f"p.grad is not None -> {p.grad is not None}")

pred.shape=(8,)  target.shape=(8, 1)
(pred - target).shape = (8, 8)   <-- 64 pairs, not 8
(pred - target.squeeze(-1)).shape = (8,)   <-- 8 pairs, correct

'loss' from the WRONG shape (averages over all 64 cross-pairs): 1.8507
loss from the RIGHT shape (averages over the intended 8 pairs):        2.0946

backward() on the WRONG shape succeeds with no error, no warning: p.grad is not None -> True


**This is the single most valuable cell in this notebook.** A `(N,1)` label tensor is not an
exotic input — it is the default output shape of a huge number of common patterns (a `DataFrame`
column, a `Linear(..., 1)` output kept unsqueezed, a mask). The failure mode is not a crash, not a
NaN, not even a suspicious number: it is a *plausible, decreasing* loss computed over the wrong
set of pairs entirely.

## 1.2 Vectorization — same computation, two ways, timed

Not every loop is a mistake, but knowing the actual cost of one is the only way to decide.

In [3]:
N = 200_000
a = torch.randn(N)
b = torch.randn(N)

t0 = time.perf_counter()
out_loop = torch.empty(N)
for i in range(N):
    out_loop[i] = a[i] * b[i] + 1.0
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
out_vec = a * b + 1.0
t_vec = time.perf_counter() - t0

print(f"Python loop:     {t_loop:.4f}s")
print(f"vectorized:      {t_vec:.6f}s")
print(f"speedup:         {t_loop/t_vec:,.0f}x")
print(f"results match:   {torch.allclose(out_loop, out_vec)}")

Python loop:     0.6125s
vectorized:      0.000245s
speedup:         2,502x
results match:   True


**Where the loop is still the right call.** Anything genuinely sequential (each step depends
on the *previous output*, like an RNN cell written by hand, or early-stopping logic that inspects
a running value) cannot be vectorized away, and forcing it into a tensor op usually just moves the
loop into confusing index arithmetic. Vectorize data-parallel work; loop over control flow.

## 1.3 dtype — silent promotion, and why float64 is special on MPS

Mixed-dtype arithmetic silently promotes to the wider type — no error, no warning, and on Apple
Silicon this specific promotion is the root of a real bug in Part 2.

In [4]:
a32 = torch.tensor([1.0], dtype=torch.float32)
b64 = torch.tensor([1.0], dtype=torch.float64)
c16 = torch.tensor([1.0], dtype=torch.float16)

print(f"float32 + float64 -> dtype {(a32 + b64).dtype}   (silently promoted to float64)")
print(f"float32 + float16 -> dtype {(a32 + c16).dtype}   (silently promoted to float32)")
print()

# float16 loses precision silently -- no error, just a wrong-looking "exact" number
big = torch.tensor([65504.0], dtype=torch.float16)  # near float16's max
print(f"float16 max-ish value {big.item()}, plus 1.0 -> {(big + 1.0).item()}  "
      f"(rounds away, no warning)")
print()

print(f"MPS available: {MPS_AVAILABLE}")
if MPS_AVAILABLE:
    try:
        _ = torch.randn(2, dtype=torch.float64, device="mps")
    except Exception as e:
        print(f"creating a float64 tensor directly ON mps: {type(e).__name__}")
        print(f"  {str(e)[:150]}")
    try:
        _ = torch.randn(2, dtype=torch.float64).to("mps")
    except Exception as e:
        print(f"moving an existing float64 CPU tensor TO mps: {type(e).__name__}")
        print(f"  {str(e)[:150]}")

float32 + float64 -> dtype torch.float64   (silently promoted to float64)
float32 + float16 -> dtype torch.float32   (silently promoted to float32)

float16 max-ish value 65504.0, plus 1.0 -> 65504.0  (rounds away, no warning)

MPS available: True
creating a float64 tensor directly ON mps: TypeError
  Cannot convert a MPS Tensor to float64 dtype as the MPS framework doesn't support float64. Please use float32 instead.
moving an existing float64 CPU tensor TO mps: TypeError
  Cannot convert a MPS Tensor to float64 dtype as the MPS framework doesn't support float64. Please use float32 instead.


**Both of the above are loud, immediate crashes — not silent.** That is worth knowing on its
own: MPS refuses float64 outright rather than quietly downcasting it, which is the *good* outcome
compared to a silent truncation. The genuinely dangerous version of this shows up in Part 2 item
1 — not the crash itself, but a *fix* for the crash that looks plausible and is not automatically
correct, needing its own proof.

## 1.4 Device semantics — what `.to()` actually does, and why order matters

In [5]:
x_cpu = torch.randn(3)

same = x_cpu.to("cpu")
print(f".to() to the SAME device you're already on: returns the same object? {same is x_cpu}")

if MPS_AVAILABLE:
    moved = x_cpu.to("mps")
    print(f".to() to a DIFFERENT device: returns the same object? {moved is x_cpu}  "
          f"(always a copy)")
    moved_again = moved.to("mps")
    print(f".to() to the device it's already on: returns the same object? {moved_again is moved}")

print()
print("Why the ORDER of .float() and .to(device) matters, not just whether both happen:")
print("  arr.to(device)[t].float()   -- if arr is float64 and device is MPS: crashes at .to()")
print("  arr.float().to(device)[t]  -- casts to float32 FIRST, then the transfer succeeds")
print("This exact reorder is the real fix in Part 2 item 1 -- and 'it runs now' is not the same")
print("claim as 'it computes the same thing' -- Part 3 has the habit that tells them apart")

.to() to the SAME device you're already on: returns the same object? True
.to() to a DIFFERENT device: returns the same object? False  (always a copy)
.to() to the device it's already on: returns the same object? True

Why the ORDER of .float() and .to(device) matters, not just whether both happen:
  arr.to(device)[t].float()   -- if arr is float64 and device is MPS: crashes at .to()
  arr.float().to(device)[t]  -- casts to float32 FIRST, then the transfer succeeds
This exact reorder is the real fix in Part 2 item 1 -- and 'it runs now' is not the same
claim as 'it computes the same thing' -- Part 3 has the habit that tells them apart


## 1.5 `view` vs `reshape`, and contiguity

In [6]:
x = torch.randn(4, 3)
xt = x.t()  # transpose -- changes strides, not memory layout
print(f"xt.is_contiguous(): {xt.is_contiguous()}")

try:
    xt.view(-1)
except RuntimeError as e:
    print(f"xt.view(-1) -- RuntimeError (loud, and it tells you what to do instead):")
    print(f"  {str(e)[:160]}")

xr = xt.reshape(-1)
print(f"xt.reshape(-1) -- succeeds silently by copying when needed: "
      f"shares memory with xt? {xr.data_ptr() == xt.data_ptr()}")

xt.is_contiguous(): False
xt.view(-1) -- RuntimeError (loud, and it tells you what to do instead):
  view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
xt.reshape(-1) -- succeeds silently by copying when needed: shares memory with xt? False


`view` refuses when the requested shape is not expressible as a reinterpretation of the
existing memory layout — a loud, informative error. `reshape` silently does whichever of the two
is needed (a view when possible, a copy when not), which is convenient and also means it will not
tell you when your code just got slower or started using twice the memory for a large tensor.

## 1.6 `model.eval()` vs `torch.no_grad()` — different things, usually both needed

In [7]:
m = nn.Sequential(nn.Linear(4, 4), nn.Dropout(0.9))
x = torch.randn(100, 4)

torch.manual_seed(0)
m.train()
with torch.no_grad():
    a1, a2 = m(x), m(x)
print(f"train() + no_grad(): two identical calls give the same output? {torch.equal(a1, a2)}  "
      f"(no_grad only stops gradient TRACKING -- dropout is still random)")

m.eval()
with torch.no_grad():
    b1, b2 = m(x), m(x)
print(f"eval()  + no_grad(): two identical calls give the same output? {torch.equal(b1, b2)}")

m.eval()
c = m(x)  # eval() alone, no no_grad()
print(f"eval() WITHOUT no_grad(): output still tracks gradients? {c.requires_grad}  "
      f"(wastes memory building a graph nobody will call .backward() on)")

train() + no_grad(): two identical calls give the same output? False  (no_grad only stops gradient TRACKING -- dropout is still random)
eval()  + no_grad(): two identical calls give the same output? True
eval() WITHOUT no_grad(): output still tracks gradients? True  (wastes memory building a graph nobody will call .backward() on)


**`model.eval()`** changes *layer behavior* (dropout off, batchnorm uses running stats).
**`torch.no_grad()`** changes *autograd bookkeeping* (stop building a graph). Neither implies the
other. Inference code needs both: `eval()` for correct outputs, `no_grad()` for correct memory use.

# Part 2 — The silent-failure catalogue

Seven specimens this project actually hit, at real cost, plus two general PyTorch gotchas kept
because they reproduced cleanly under direct testing. Several candidates were tried and dropped —
see the note at the end of this Part for what was tried and why it didn't make the cut.

## 2.1 float64 shipped to MPS before casting (`docs/LANDMINES.md` §7, §22)

MDM's diffusion schedules (`sqrt_alphas_cumprod` etc.) are built as **numpy float64** arrays.
`_extract_into_tensor` originally did `th.from_numpy(arr).to(device)[t].float()` — move first,
cast after. On MPS this crashes immediately (Part 1.3). The fix reorders to
`.float().to(device)[t]` — cast first. **A crash going away is not proof the fix computes the
same thing** — that has to be checked separately.

In [8]:
arr = np.random.RandomState(0).rand(1000).astype(np.float64)  # a schedule array, like MDM's
t = torch.randint(0, 1000, (32,))

# the buggy order (simulated on CPU, since MPS would just crash here)
buggy = torch.from_numpy(arr).to("cpu")[t].float()
# the fixed order
fixed = torch.from_numpy(arr).float().to("cpu")[t]

print(f"buggy-order and fixed-order results identical? {torch.equal(buggy, fixed)}")
print(f"WHY they must be identical, not just close: indexing (arr[t]) is a pure gather -- no")
print(f"arithmetic happens between the cast and the gather in either order, so cast-then-gather")
print(f"and gather-then-cast commute exactly. This is provable, not merely empirically similar.")

buggy-order and fixed-order results identical? True
WHY they must be identical, not just close: indexing (arr[t]) is a pure gather -- no
arithmetic happens between the cast and the gather in either order, so cast-then-gather
and gather-then-cast commute exactly. This is provable, not merely empirically similar.


## 2.2 A device-selection function written for two devices, silently skipping a third

Contrast with 2.1: this one produces **no error at all**.

In [9]:
def dev_BUGGY():
    '''Written when only CUDA and CPU existed.'''
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

def dev_FIXED():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

print(f"MPS is available on this machine: {MPS_AVAILABLE}")
print(f"dev_BUGGY() returns: {dev_BUGGY()}   <-- wrong, and nothing about this call says so")
print(f"dev_FIXED() returns: {dev_FIXED()}")
print()
print("The buggy version doesn't crash, doesn't warn, doesn't run measurably differently for a")
print("tiny example -- it just quietly leaves a 5-10x speedup on the table, forever, until")
print("someone happens to print the device a training run actually used.")

MPS is available on this machine: True
dev_BUGGY() returns: cpu   <-- wrong, and nothing about this call says so
dev_FIXED() returns: mps

The buggy version doesn't crash, doesn't warn, doesn't run measurably differently for a
tiny example -- it just quietly leaves a 5-10x speedup on the table, forever, until
someone happens to print the device a training run actually used.


## 2.3 One fix, verified correct, is not the same as the bug being gone everywhere

Section 2.1's cast-after-transfer defect was found and fixed in
`diffusion/gaussian_diffusion.py`. **The identical anti-pattern was independently present in a
second, unrelated file** (`data_loaders/humanml/networks/evaluator_wrapper.py`,
`.detach().to(self.device).float()`), reached only by a completely different call path (scoring a
generated motion, not diffusing one). A validation that had been argued to be redundant was run
anyway — and it found this second instance.

In [10]:
import subprocess
# grep this project's own vendored code for the anti-pattern directly, rather than asserting
# from memory that it existed in exactly two places
result = subprocess.run(
    ["grep", "-rn", r"\.to(self\.device)\.float()\|\.to(device)\[.*\]\.float()",
     "../third_party/motion-diffusion-model/"],
    capture_output=True, text=True
)
print("Searching the vendored codebase for the SAME buggy pattern:")
print(result.stdout if result.stdout else "(no matches)")

# is any matched file actually IMPORTED anywhere on this project's own execution path?
# (search for a real import statement, not just the name as a substring -- and exclude this
# notebook's own .ipynb file, which necessarily contains these filenames as text right now)
for module in ["trainers", "simplify_loc2rot"]:
    found = subprocess.run(
        ["grep", "-rl", "--include=*.py",
         f"import {module}\\|from.*{module} import\\|networks.trainers\\|visualize.{module}",
         "../scripts/", "../demo/"],
        capture_output=True, text=True
    ).stdout
    print(f"{module}.py: imported by a real import statement anywhere in this project's own "
          f"scripts/demo (.py files only, not this notebook's own text)? "
          f"{'yes -- ' + found.strip().replace(chr(10), ', ') if found.strip() else 'no'}")

Searching the vendored codebase for the SAME buggy pattern:
../third_party/motion-diffusion-model/visualize/simplify_loc2rot.py:75:        keypoints_3d = torch.Tensor(input_joints).to(self.device).float()
../third_party/motion-diffusion-model/data_loaders/humanml/networks/trainers.py:56:        self.motions = motions.detach().to(self.device).float()
../third_party/motion-diffusion-model/data_loaders/humanml/networks/trainers.py:279:        word_emb = word_emb.detach().to(self.device).float()
../third_party/motion-diffusion-model/data_loaders/humanml/networks/trainers.py:280:        pos_ohot = pos_ohot.detach().to(self.device).float()
../third_party/motion-diffusion-model/data_loaders/humanml/networks/trainers.py:281:        motions = motions.detach().to(self.device).float()
../third_party/motion-diffusion-model/data_loaders/humanml/networks/trainers.py:383:        word_emb = word_emb.detach().to(self.device).float()
../third_party/motion-diffusion-model/data_loaders/humanml/networks/tr

**Reading this cell's real output, not an assumed one.** The grep finds more than the two
instances this section opened with — `trainers.py` (already known and flagged,
`docs/DECISIONS.md`, since it only retrains the evaluator networks and this project never
retrains them — pretrained checkpoints are loaded, unmodified) and, newly found while writing
this cell, `visualize/simplify_loc2rot.py` (MDM's optional SMPL-mesh rendering path — confirmed
above to be imported nowhere in this project's own code, which renders skeletons its own way,
`notebooks/03`). **Both remaining instances are genuinely unexercised here, not silently
dangerous** — but "unexercised, confirmed" is a specific, checked claim, not the same as "doesn't
exist," and the check above is what makes the difference visible rather than assumed.

**The lesson, not just the bug.** Passing a check that covers *one* code path licenses nothing
about a *different* code path, even one that looks structurally identical, if the two are reached
independently. "This class of bug is fixed" is a claim about specific call sites, not about the
codebase in general — Part 3's device-parity habit exists precisely to make that claim checkable
rather than assumed.

## 2.4 Sorting an already-sorted array again can silently swap tied elements

**This section was rewritten after this notebook's own construction found a fourth bug in
`notebooks/02`, previously believed settled twice.** The original plan for this section was to
demonstrate "motion embeddings are not batch-composition-invariant" as an established project
finding. Investigating *why*, directly, found instead that the claim itself was an artifact.
`docs/LANDMINES.md` §25 has the full account; this is the demonstrable core of it.

In [11]:
from torch.nn.utils.rnn import pack_padded_sequence as pack

class TinyEncoder(nn.Module):
    '''Mimics the real shape of the bug: a function that sorts its input by length
    internally (a pack_padded_sequence requirement) and does NOT un-sort before returning --
    exactly MDM's own EvaluatorMDMWrapper.get_motion_embeddings, by its own docstring.'''
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(4, 6, batch_first=True)

    def forward(self, x, lens):
        align_idx = np.argsort(lens)[::-1].copy()   # descending sort, INTERNAL
        x_sorted = x[align_idx]
        lens_sorted = [lens[i] for i in align_idx]
        packed = pack(x_sorted, lens_sorted, batch_first=True)
        _, h = self.gru(packed)
        return h[0]   # returned in align_idx order -- NOT the caller's input order

enc = TinyEncoder()
enc.eval()

torch.manual_seed(3)
x = torch.randn(6, 5, 4)
lens = [3, 3, 5, 3, 2, 3]   # lots of ties, like this project's own real generated-motion lengths

def call_CORRECTLY(x, lens):
    align_idx = np.argsort(lens)[::-1].copy()
    inv = np.argsort(align_idx)
    with torch.no_grad():
        out_sorted = enc(x, lens)
    return out_sorted[inv]           # invert the ONE sort the function itself did

def call_WITH_DOUBLE_SORT(x, lens):
    order = np.argsort(lens)[::-1].copy()   # an EXTERNAL pre-sort, seems reasonable
    inv = np.argsort(order)
    with torch.no_grad():
        out = enc(x[order], [lens[i] for i in order])   # enc() sorts AGAIN, internally
    return out[inv]                          # only undoes the outer sort, not enc's own

correct = call_CORRECTLY(x, lens)
buggy = call_WITH_DOUBLE_SORT(x, lens)

diff = (correct - buggy).abs().max(dim=1).values
print(f"per-sample max abs diff, correct vs. double-sorted call:")
for i, d in enumerate(diff.tolist()):
    print(f"  sample {i} (length={lens[i]}): {d:.4f}" + ("  <-- swapped!" if d > 1e-4 else ""))

per-sample max abs diff, correct vs. double-sorted call:
  sample 0 (length=3): 0.1396  <-- swapped!
  sample 1 (length=3): 0.6244  <-- swapped!
  sample 2 (length=5): 0.0000
  sample 3 (length=3): 0.6244  <-- swapped!
  sample 4 (length=2): 0.0000
  sample 5 (length=3): 0.1396  <-- swapped!


**Reading this.** Samples with a *unique* length are unaffected (there is nothing to
tie-break). Samples sharing a length with another sample in the batch can come out swapped,
because numpy's default sort is not stable, and the outer inversion only undoes the *outer* sort
— it has no visibility into the *inner* function's own, independent tie-break. In this project's
own real data, **120 of 128 generated motions share their length with at least one other
sample** — ties were the norm, not an edge case, and this cost a headline number: `notebooks/02`'s
Guo R-Precision-top3 moved from 0.7266 to an exact 0.7578 once the double-sort was removed —
matching this project's own independently recorded target to the digit, not merely landing
"within noise" as the notebook had twice concluded before this was found.

**The corrected general claim:** call a function that does its own internal sorting exactly
once, invert exactly that one sort, and never sort its input yourself beforehand. Under that
calling convention, this project's real motion encoder is batch-composition-invariant, exactly
like its text encoder — the earlier claim that it wasn't wasn't a property of the network. It was
this same bug, in an earlier form.

## 2.5 Fitting data normalized for training into an evaluator expecting raw features

Every number stays plausible. Nothing crashes. The scores are simply computed on data in the
wrong statistical space.

In [12]:
# a toy "evaluator" that (like the real Guo evaluator) expects RAW features and applies its
# OWN normalization internally
class ToyEvaluator:
    def __init__(self, expected_mean, expected_std):
        self.mean, self.std = expected_mean, expected_std
    def score(self, raw_features):
        normed = (raw_features - self.mean) / self.std   # evaluator's OWN normalization
        return normed.abs().mean().item()  # a stand-in "quality score"

torch.manual_seed(4)
raw = torch.randn(1000) * 3.0 + 5.0                 # true feature distribution: mean 5, std 3
train_mean, train_std = raw.mean(), raw.std()        # the TRAINING pipeline's own normalization
training_normalized = (raw - train_mean) / train_std  # what a generator's OWN diffusion sampling
                                                       # actually operates on and outputs

evaluator = ToyEvaluator(expected_mean=raw.mean(), expected_std=raw.std())  # evaluator wants RAW

score_correct = evaluator.score(raw)
score_bug = evaluator.score(training_normalized)  # feeding TRAINING-space data into an evaluator
                                                    # that expects RAW-space data -- no error

print(f"evaluator score on correctly-prepared (raw) data:  {score_correct:.4f}")
print(f"evaluator score on TRAINING-normalized data fed in by mistake: {score_bug:.4f}")
print(f"both numbers are plausible floats in a similar range -- nothing here looks 'wrong'")

evaluator score on correctly-prepared (raw) data:  0.7999
evaluator score on TRAINING-normalized data fed in by mistake: 1.5677
both numbers are plausible floats in a similar range -- nothing here looks 'wrong'


**The real version of this cost more than a wrong number — it flipped a conclusion.**
`notebooks/02`'s cached motions sit in MDM's own training normalization; both Guo and TMR expect
raw HumanML3D features and apply their own normalization internally. Feeding the wrong space in
moved Guo R-Precision-top3 from 0.6641 to 0.7422 in isolation, and — because the bug touched both
evaluators being compared — silently made an *uncorrelated-looking* comparison look uncorrelated
for a reason that had nothing to do with either evaluator's actual behavior.

## 2.6 Feeding an encoder tokens it was never trained on

An NLP encoder trained on one tokenization scheme does not raise an error when handed a
*different, plausible-looking* tokenization of the same sentence — it just embeds nonsense
fluently.

In [13]:
# HumanML3D's own captions are pre-tokenized and LEMMATIZED ("walk", "run") -- that is what
# the Guo text encoder in this project was actually trained on. A fresh spaCy POS-tagger returns
# SURFACE forms instead ("walking", "running") -- plausible, common, and wrong for this encoder.
lemmatized = ["a/DET", "person/NOUN", "walk/VERB", "forward/ADV"]
surface    = ["a/DET", "person/NOUN", "walking/VERB", "forward/ADV"]

print("Both token lists look like completely reasonable POS-tagged sentences.")
print("Lemmatized (what the encoder was trained on): ", lemmatized)
print("Surface form (a fresh, 'correct' POS-tagger):  ", surface)
print()
print("Real, measured cost in this project: of 128 real captions, 198 of 200 spot-checked tokens")
print("differed between the two schemes (`docs/EXPERIMENT_LOG.md`), and Guo R-Precision-top3 on")
print("the SAME 128 motions moved from 0.352 to 0.4375 from this fix alone -- no exception was")
print("ever raised; a differently-tokenized sentence is still a completely valid input.")

Both token lists look like completely reasonable POS-tagged sentences.
Lemmatized (what the encoder was trained on):  ['a/DET', 'person/NOUN', 'walk/VERB', 'forward/ADV']
Surface form (a fresh, 'correct' POS-tagger):   ['a/DET', 'person/NOUN', 'walking/VERB', 'forward/ADV']

Real, measured cost in this project: of 128 real captions, 198 of 200 spot-checked tokens
differed between the two schemes (`docs/EXPERIMENT_LOG.md`), and Guo R-Precision-top3 on
the SAME 128 motions moved from 0.352 to 0.4375 from this fix alone -- no exception was
ever raised; a differently-tokenized sentence is still a completely valid input.


## 2.7 A diagnostic that consumes RNG state changes downstream results

Adding a print statement, a logging pass, or a sanity-check loop that happens to draw from the
global random state shifts *everything after it* — with no error, and no visible connection
between the added line and the changed numbers.

In [14]:
def draw_batch(loader_size=10, batch_size=4):
    return torch.randperm(loader_size)[:batch_size]

torch.manual_seed(42)
batch_1 = draw_batch()   # a ground-truth loader's own shuffle, say

torch.manual_seed(42)
_ = torch.rand(5)        # an "innocent" diagnostic: e.g. iterating a second loader once to cache
                          # its output, exactly as this project's own E0b v2 run did
batch_2 = draw_batch()   # the SAME loader, SAME seed, but the RNG has already moved on

print(f"batch drawn with nothing in between:      {batch_1.tolist()}")
print(f"batch drawn after ONE unrelated rand() call: {batch_2.tolist()}")
print(f"identical? {torch.equal(batch_1, batch_2)}")
print()
print("This project's own E0b round 2 added exactly this kind of diagnostic (caching a second")
print("loader's motions) and its ground-truth batch composition changed as a direct result --")
print("mistaken at the time for 'the run reproduced, so results are stable' when what had")
print("actually happened was that ONE of two numbers moved for a reason unrelated to the change")
print("under test (`docs/EXPERIMENT_LOG.md`, the v1-vs-v2 retraction).")

batch drawn with nothing in between:      [2, 6, 1, 8]
batch drawn after ONE unrelated rand() call: [5, 8, 6, 3]
identical? False

This project's own E0b round 2 added exactly this kind of diagnostic (caching a second
loader's motions) and its ground-truth batch composition changed as a direct result --
mistaken at the time for 'the run reproduced, so results are stable' when what had
actually happened was that ONE of two numbers moved for a reason unrelated to the change
under test (`docs/EXPERIMENT_LOG.md`, the v1-vs-v2 retraction).


## 2.8 `torch.tensor(existing_tensor)` copies AND silently detaches

A quiet `UserWarning` most logging setups never surface, plus a real, easy-to-miss break in the
autograd graph.

In [15]:
x = torch.tensor([1.0, 2.0], requires_grad=True)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    y = torch.tensor(x)   # looks like a harmless "copy" of x
print(f"warning raised: {len(caught) > 0}")
if caught:
    print(f"  {str(caught[0].message)[:140]}")
print(f"y.requires_grad: {y.requires_grad}   (silently False -- gradients stop flowing here)")
print(f"y shares memory with x: {y.data_ptr() == x.data_ptr()}   (it's also a real copy, not a view)")
print()
print("Correct alternative, which says what it means:")
z = x.detach().clone()
print(f"x.detach().clone(): requires_grad={z.requires_grad}, same warning raised: explicit, not silent")

warning raised: True
  To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True
y.requires_grad: False   (silently False -- gradients stop flowing here)
y shares memory with x: False   (it's also a real copy, not a view)

Correct alternative, which says what it means:
x.detach().clone(): requires_grad=False, same warning raised: explicit, not silent


## 2.9 Forgetting to zero gradients silently accumulates them

Perhaps the single most common real PyTorch bug for anyone new to writing a training loop by
hand — no error, a loss that still moves, just wrong.

In [16]:
w = torch.tensor([1.0], requires_grad=True)
for step in range(3):
    loss = (w * 2).sum()
    loss.backward()   # NOTE: no w.grad.zero_() / optimizer.zero_grad() before this
    print(f"step {step}: w.grad = {w.grad.item()}  "
          f"{'<-- should be 2.0 every step, but it is accumulating' if step > 0 else ''}")

step 0: w.grad = 2.0  
step 1: w.grad = 4.0  <-- should be 2.0 every step, but it is accumulating
step 2: w.grad = 6.0  <-- should be 2.0 every step, but it is accumulating


**Why this matters more than it looks.** A step count of gradient values that silently
grows makes an optimizer take an ever-larger step each iteration — often still *looking* like
training (loss moves, sometimes even usefully for a while) before it diverges or plateaus in a
way nothing points back to this specific line.

## What was tried and dropped

Per the standing instruction for this notebook: verify every phenomenon directly, and say so when
one does not reproduce, rather than keeping a plausible-sounding claim untested. Each one below
is tested in its own cell, not just asserted in prose.

In [17]:
# Candidate 1: pack_padded_sequence with unsorted lengths (enforce_sorted=True, the default)
x = torch.randn(3, 5, 2)
lens_unsorted = [2, 5, 3]  # NOT descending
try:
    pack_padded_sequence(x, lens_unsorted, batch_first=True)
    print("no error -- would have been a silent failure candidate")
except RuntimeError as e:
    print(f"RuntimeError (loud, immediate, tells you exactly what to do instead):")
    print(f"  {str(e)[:180]}")
print("DROPPED as its own catalogue item: this is a loud crash, not a silent failure. Kept as")
print("the API 2.4's toy above deliberately works around by sorting correctly.")

RuntimeError (loud, immediate, tells you exactly what to do instead):
  `lengths` array must be sorted in decreasing order when `enforce_sorted` is True. You can pass `enforce_sorted=False` to pack_padded_sequence and/or pack_sequence to sidestep this 
DROPPED as its own catalogue item: this is a loud crash, not a silent failure. Kept as
the API 2.4's toy above deliberately works around by sorting correctly.


In [18]:
# Candidate 2: argsort tie-breaking, CPU vs MPS
vals = torch.tensor([3.0, 1.0, 1.0, 2.0, 1.0, 1.0, 5.0, 1.0])  # several ties
cpu_order = torch.argsort(vals, descending=True)
print(f"CPU argsort (descending) of a tied tensor: {cpu_order.tolist()}")
if MPS_AVAILABLE:
    mps_order = torch.argsort(vals.to("mps"), descending=True).cpu()
    print(f"MPS argsort (descending), same tensor:     {mps_order.tolist()}")
    print(f"identical tie-break order? {torch.equal(cpu_order, mps_order)}")
    print("DROPPED: no difference reproduced in this test, on this PyTorch version/hardware.")
else:
    print("MPS not available on this machine to compare against -- candidate untested here.")

CPU argsort (descending) of a tied tensor: [6, 0, 3, 1, 2, 4, 5, 7]
MPS argsort (descending), same tensor:     [6, 0, 3, 1, 2, 4, 5, 7]
identical tie-break order? True
DROPPED: no difference reproduced in this test, on this PyTorch version/hardware.


In [19]:
# Candidate 3: in-place ops breaking autograd (the classic textbook construction)
a = torch.randn(3, requires_grad=True)
b = a * 2
c = b * 3
try:
    b += 1          # in-place on b, used by c's backward -- the textbook "this should break" case
    c.sum().backward()
    print("no error, and gradient computed -- because c = b*3's backward needs only the constant")
    print("3, not b's stored VALUE, so modifying b in place doesn't corrupt this particular graph.")
except RuntimeError as e:
    print(f"RuntimeError: {str(e)[:200]}")
print("DROPPED as its own item: didn't fail for this op sequence. Replaced by 2.9 (gradient")
print("accumulation), a different, more common, and cleanly reproducible failure nearby.")

no error, and gradient computed -- because c = b*3's backward needs only the constant
3, not b's stored VALUE, so modifying b in place doesn't corrupt this particular graph.
DROPPED as its own item: didn't fail for this op sequence. Replaced by 2.9 (gradient
accumulation), a different, more common, and cleanly reproducible failure nearby.


# Part 3 — The habits that catch these

Each one below is a short, reusable snippet, not a one-off fix. The five habits above cover every
specimen in Part 2.

## 3.1 Shape assertions at function boundaries

Would have caught 1.1 and 2.4 (a swapped-but-plausible shape) immediately.

In [20]:
def compute_loss(pred, target):
    assert pred.shape == target.shape, (
        f"shape mismatch: pred {tuple(pred.shape)} vs target {tuple(target.shape)} -- "
        f"broadcasting would silently produce {tuple(torch.broadcast_shapes(pred.shape, target.shape))}"
    )
    return ((pred - target) ** 2).mean()

pred = torch.randn(8)
target_wrong = torch.randn(8, 1)
try:
    compute_loss(pred, target_wrong)
except AssertionError as e:
    print(f"caught before it could silently broadcast: {e}")

caught before it could silently broadcast: shape mismatch: pred (8,) vs target (8, 1) -- broadcasting would silently produce (8, 8)


## 3.2 Determinism check — run twice, compare bitwise

In [21]:
def determinism_check(fn, *args, **kwargs):
    out1 = fn(*args, **kwargs)
    out2 = fn(*args, **kwargs)
    identical = torch.equal(out1, out2) if torch.is_tensor(out1) else out1 == out2
    print(f"{fn.__name__}: identical across two runs? {identical}")
    return identical

m = nn.Sequential(nn.Linear(4, 4), nn.Dropout(0.5))
m.eval()  # correctly set up for a determinism check -- see Part 1.6
x = torch.randn(10, 4)
with torch.no_grad():
    determinism_check(lambda: m(x))

<lambda>: identical across two runs? True


## 3.3 Batch-invariance check — this project's own invented technique

**Encode one sample alone, and again as part of a larger batch; compare.** This project
discovered the need for this check by accident (chasing a different acceptance test) and it is
general enough to name: any function whose per-sample output should not depend on its
batch-mates should pass this check before being trusted for anything comparative.

In [22]:
def batch_invariance_check(encode_fn, x, sample_idx=0):
    alone = encode_fn(x[sample_idx:sample_idx+1])
    in_batch = encode_fn(x)[sample_idx:sample_idx+1]
    diff = (alone - in_batch).abs().max().item()
    print(f"sample {sample_idx} alone vs. inside a batch of {x.shape[0]}: max abs diff = {diff:.6f}"
          f"  {'OK' if diff < 1e-4 else '<-- INVESTIGATE'}")
    return diff

lin = nn.Linear(4, 4)
lin.eval()
x = torch.randn(16, 4)
with torch.no_grad():
    batch_invariance_check(lambda t: lin(t), x)

sample 0 alone vs. inside a batch of 16: max abs diff = 0.000000  OK


## 3.4 Cross-check against a number you already trust

**Not code review, not a test suite — this specific habit is what caught all three of this
project's original `notebooks/02` bugs, and the fourth (section 2.4 above) besides.** Before
trusting a new computation's output, compute the same quantity a second, independent way, or
compare against a previously-recorded number for the identical input.

In [23]:
def cross_check(new_value, trusted_value, tolerance=0.01, label=""):
    gap = abs(new_value - trusted_value)
    ok = gap <= tolerance
    print(f"{label}: new={new_value:.4f}  trusted={trusted_value:.4f}  gap={gap:.4f}  "
          f"{'within tolerance' if ok else '<-- INVESTIGATE, do not proceed on this number'}")
    return ok

cross_check(0.7578, 0.7578, label="notebooks/02's own real acceptance check, after all four fixes")

notebooks/02's own real acceptance check, after all four fixes: new=0.7578  trusted=0.7578  gap=0.0000  within tolerance


True

## 3.5 Device-parity gate — before trusting a new device, reproduce a known result on it

**Before running anything real on a newly-enabled device, reproduce a result you already trust
there, and compare the device-to-device delta against the ordinary seed-to-seed delta.** If the
device delta is not small relative to seed noise, nothing run on that device yet deserves trust.

In [24]:
def device_parity_gate(known_good_value, device_value, seed_noise_estimate, label=""):
    device_delta = abs(known_good_value - device_value)
    ratio = device_delta / seed_noise_estimate if seed_noise_estimate > 0 else float("inf")
    passed = ratio < 1.0
    print(f"{label}")
    print(f"  device delta:      {device_delta:.2e}")
    print(f"  seed-to-seed noise: {seed_noise_estimate:.2e}")
    print(f"  device delta is {ratio:.4f}x the seed noise -- "
          f"{'device is trustworthy for this computation' if passed else 'DO NOT TRUST YET'}")
    return passed

# this project's own real E0a gate (docs/DECISIONS.md D-27): MPS-vs-CPU delta on a reproduced
# ground-truth number, compared against this project's own measured seed-to-seed spread
device_parity_gate(known_good_value=0.7969, device_value=0.7969 + 4e-8,
                    seed_noise_estimate=2.7e-3, label="MPS vs. CPU, ground-truth R-Precision reproduction")

MPS vs. CPU, ground-truth R-Precision reproduction
  device delta:      4.00e-08
  seed-to-seed noise: 2.70e-03
  device delta is 0.0000x the seed noise -- device is trustworthy for this computation


True

## What it means

None of the nine specimens in Part 2 announced themselves. Every one produced a number that was
plausible on its own — in several cases, *more* plausible-looking than the correct answer (a
"cleaner" all-at-once embedding call; a fresh POS-tagger's grammatically correct tokens; a loss
that kept decreasing while accumulating gradients). **Plausibility is not evidence.** The five
habits in Part 3 are cheap, general, and were each sufficient to catch at least one of these
specimens directly — they do not require knowing the specific bug in advance, which is the entire
point of a habit over a fix.

**The specimen that changed the most while writing this notebook (section 2.4) is itself the
best argument for Part 3.4.** A cross-check against a trusted number — this project's own prior
0.7578 — is what turned "these two evaluators agree, roughly" into "these two evaluators agree
exactly, once a real bug is found," four bugs into a notebook that had twice already seemed
finished.

## What would change my mind

- **This notebook's toy specimens are deliberately small.** Several (2.4, 2.5, 2.7) are minimal
  reconstructions of a real project bug, not the original buggy code itself — chosen to isolate
  the mechanism, not to be a byte-for-byte replay. The real numbers cited alongside each (from
  `docs/LANDMINES.md` and `notebooks/02`) are the actual measurements; the toy cells are there so
  the mechanism can be re-run and inspected directly, not taken on faith.
- **"Dropped, did not reproduce" (argsort tie-breaking, classic in-place autograd corruption) is
  reported honestly as a negative result for this specific test, on this specific PyTorch version
  and hardware** — not as proof the phenomenon can never occur under any configuration. A
  different PyTorch version, a different op sequence, or genuinely divergent MPS/CPU kernels
  could still surface either one; this notebook only reports what was actually tried.
- **Section 2.3's grep does not show zero matches, and it shouldn't be read as if it should.**
  It finds two more live instances of the anti-pattern (`trainers.py`, `simplify_loc2rot.py`),
  both confirmed, in the same cell, to be unreachable from this project's own code. The check's
  value was never "prove the pattern is gone from the codebase" — it was "prove this specific,
  reachable instance is fixed, and name what else matches without assuming it matters."
- **If a future PyTorch or MPS release changes any of these behaviors** (fixes float64-on-MPS,
  changes `argsort`'s stability guarantees, or alters autograd's in-place versioning), the
  specific numbers here would need re-verification — the habits in Part 3 would not, since they
  do not depend on any of these specifics remaining true.